# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print("Dataset Name:", getattr(dataset.metadata, 'name', 'Unknown'))
print("Description:", getattr(dataset.metadata, 'description', 'No description'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Extract record set list and info
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for i, rs in enumerate(record_sets):
    print(f"{i+1}. RecordSet @id: {rs['@id']}")
    print(f"   Name:    {rs.get('name', 'N/A')}")
    print(f"   Description: {rs.get('description', 'N/A')}")
    # List fields
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"   Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"     - {field.get('@id', '--')} (name: {field.get('name', '--')})")
        else:
            print(f"     - {field}")
    print()
# Save first record set id for following code blocks
first_record_set_id = record_sets[0]['@id'] if len(record_sets) else None

You can display the first few records for the primary record set (replace `first_record_set_id` if desired):

In [ ]:
# Print some sample records from the first record set
if first_record_set_id is not None:
    print(f"First 2 records from record set {first_record_set_id}:")
    records_iter = dataset.records(record_set=first_record_set_id)
    for i, record in enumerate(records_iter):
        print(record)
        if i >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record set IDs
all_record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# List columns for the main record set
if first_record_set_id is not None:
    print(f"Columns in DataFrame for RecordSet {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# You may need to inspect numeric fields. We'll do so for the first record set.
df = dataframes[first_record_set_id].copy()

# Display data types and try to identify a numeric field
print("Data types:")
print(df.dtypes)
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print("Numeric candidates:", numeric_candidates)

# For the demonstration, choose the first numeric column
numeric_field = numeric_candidates[0] if len(numeric_candidates) else None
if numeric_field:
    print(f"Proceeding with EDA on column: {numeric_field}")
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    
    # Normalize
    filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())
    
    # Group by a field, if a categorical/nominal field is present
    non_numeric_candidates = df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for col in non_numeric_candidates:
        if df[col].nunique() < 10:
            group_field = col
            break
    if group_field:
        print(f"Grouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        display(grouped_df.head())
else:
    print("No numeric fields were found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored dataset metadata and identified available record sets and fields (referenced by their `@id`).
- Loaded data into DataFrames using `mlcroissant`.
- Demonstrated basic EDA and normalization techniques on a numeric field.
- Visualized data distributions to understand feature properties.

Further analysis can include deeper feature engineering, advanced visualizations, and integration with model building workflows.